# Real-ESRGAN Logo Hazirlama / QR Fusion Oncesi Pipeline

Bu notebook musteri logosunu lokal bilgisayardan upload eder, analiz eder, Real-ESRGAN ile upscale eder ve QR fusion icin temiz kare PNG ciktilar hazirlar.

Varsayilan ayarlar ticari logo/ikon isleri ve A100 GPU icin secildi: anime/illustration modeli, 4x upscale, 3072 hedef boyut, contain padding, beyaz arka plan, hafif keskinlestirme.

In [ ]:
#@title 1) Kurulum / Repo Hazirlama
import os, sys, subprocess, textwrap, shutil

PROJECT_DIR = "/content/Real-ESRGAN"
USE_FRESH_CLONE = True #@param {type:"boolean"}

os.chdir('/content')
if USE_FRESH_CLONE and os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
if not os.path.isdir(PROJECT_DIR):
    !git clone https://github.com/xinntao/Real-ESRGAN.git

%cd /content/Real-ESRGAN

# Colab ortaminda kurulum. Runtime zaten torch+cuda ile gelir; onu yeniden kurmuyoruz.
!pip install -q basicsr facexlib gfpgan opencv-python Pillow tqdm
!pip install -q -e .

# Yeni torchvision surumlerinde basicsr'in bekledigi eski modul eksikse kucuk bir uyumluluk shim'i ekle.
try:
    import torchvision.transforms.functional_tensor
except Exception:
    import site, pathlib
    site_dir = pathlib.Path(site.getsitepackages()[0])
    shim = site_dir / 'torchvision' / 'transforms' / 'functional_tensor.py'
    shim.write_text('from .functional import rgb_to_grayscale\n')
    print('torchvision functional_tensor shim eklendi:', shim)

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
#@title 2) Logo Upload
from google.colab import files
from pathlib import Path
import os, shutil

UPLOAD_DIR = Path('/content/logo_upload')
WORK_DIR = Path('/content/logo_work')
OUTPUT_DIR = Path('/content/logo_outputs')
for p in [UPLOAD_DIR, WORK_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PNG, JPG, JPEG veya WEBP logo dosyanizi secin.')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Dosya yuklenmedi.')

first_name = next(iter(uploaded.keys()))
src = UPLOAD_DIR / first_name
with open(src, 'wb') as f:
    f.write(uploaded[first_name])

print('Yuklenen logo:', src)
LOGO_PATH = str(src)

In [ ]:
#@title 3) Logo Analiz
from PIL import Image, ImageStat, ImageFilter
from pathlib import Path
import numpy as np, os, json, math
from IPython.display import display

img = Image.open(LOGO_PATH)
img_for_stats = img.convert('RGBA')
arr = np.array(img_for_stats)
rgb = arr[:, :, :3]
alpha = arr[:, :, 3]
w, h = img.size
has_alpha = img.mode in ('RGBA', 'LA') or ('transparency' in img.info)
transparent_pixels = int((alpha < 250).sum())
transparent_ratio = transparent_pixels / float(alpha.size)
is_transparent = has_alpha and transparent_ratio > 0.01
white_ratio = float(((rgb > 245).all(axis=2)).sum()) / float(w * h)
gray = img.convert('L')
contrast = float(ImageStat.Stat(gray).stddev[0])
small_side = min(w, h)
lap = np.asarray(gray.filter(ImageFilter.FIND_EDGES), dtype=np.float32)
edge_score = float(lap.mean())

warnings = []
if small_side < 512:
    warnings.append('Logo 512x512 altinda, 4x upscale onerilir.')
if not is_transparent:
    if white_ratio > 0.35:
        warnings.append('Logo seffaf degil; beyaz arka plan agirlikli gorunuyor.')
    else:
        warnings.append('Logo seffaf degil; QR fusion icin solid arka planla hazirlanacak.')
if contrast < 28:
    warnings.append('Kontrast dusuk; QR fusion okunabilirligi zorlasabilir.')
if edge_score < 4:
    warnings.append('Goruntu yumusak/bulanik olabilir; sharpen faydali olur.')
if w < 700 or h < 700:
    warnings.append('Kucuk detay veya yazi varsa finalde bozulabilir.')

print('Format:', img.format)
print('Mode:', img.mode)
print('Boyut:', f'{w}x{h}')
print('Seffaflik:', 'var' if is_transparent else 'yok')
print('Beyaz piksel orani:', round(white_ratio, 3))
print('Kontrast skoru:', round(contrast, 2))
print('Keskinlik/kenar skoru:', round(edge_score, 2))
print('\nUyarilar:')
if warnings:
    for item in warnings:
        print('-', item)
else:
    print('- Logo upscale icin uygun gorunuyor.')

display(img.resize((min(512, w), int(h * min(512, w) / w))))

In [ ]:
#@title 4) Ayarlar
upscale_enabled = True #@param {type:"boolean"}
upscale_model = "realesrgan-x4plus-anime" #@param ["realesrgan-x4plus-anime", "realesrgan-x4plus", "realesrgan-x2plus", "realesr-general-x4v3"]
upscale_factor = 4 #@param [2, 4] {type:"raw"}
target_size = 3072 #@param [2048, 3072, 4096] {type:"raw"}
fit_mode = "contain_pad" #@param ["contain_pad", "cover_crop"]
background_mode = "white" #@param ["white", "keep_transparent", "custom_color"]
custom_background_color = "#FFFFFF" #@param {type:"string"}
denoise_strength = 0.2 #@param {type:"slider", min:0, max:1, step:0.05}
sharpen_after = 1.2 #@param {type:"slider", min:1, max:1.8, step:0.05}
tile_size = 768 #@param [0, 512, 768, 1024] {type:"raw"}
output_format = "png" #@param ["png"]

MODEL_MAP = {
    'realesrgan-x4plus-anime': 'RealESRGAN_x4plus_anime_6B',
    'realesrgan-x4plus': 'RealESRGAN_x4plus',
    'realesrgan-x2plus': 'RealESRGAN_x2plus',
    'realesr-general-x4v3': 'realesr-general-x4v3',
}

print('Secilen model:', upscale_model)
print('Real-ESRGAN model adi:', MODEL_MAP[upscale_model])
print('Hedef:', f'{target_size}x{target_size}', '| fit:', fit_mode, '| background:', background_mode)

In [ ]:
#@title 5) Real-ESRGAN Upscale Calistir
from pathlib import Path
from PIL import Image
import subprocess, os, shutil, glob

WORK_DIR = Path('/content/logo_work')
ESR_INPUT = WORK_DIR / 'esr_input.png'
ESR_OUTPUT_DIR = Path('/content/logo_esrgan_result')
ESR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Real-ESRGAN icin inputu PNG olarak normalize et.
Image.open(LOGO_PATH).convert('RGBA').save(ESR_INPUT)

if upscale_enabled:
    cmd = [
        sys.executable, 'inference_realesrgan.py',
        '-n', MODEL_MAP[upscale_model],
        '-i', str(ESR_INPUT),
        '-o', str(ESR_OUTPUT_DIR),
        '-s', str(upscale_factor),
        '--suffix', 'upscaled',
        '--ext', 'png',
        '--alpha_upsampler', 'realesrgan',
        '-t', str(tile_size),
        '--tile_pad', '16',
        '--gpu-id', '0',
    ]
    if upscale_model == 'realesr-general-x4v3':
        cmd += ['-dn', str(denoise_strength)]
    print('Komut:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    candidates = sorted(ESR_OUTPUT_DIR.glob('*_upscaled.png'))
    if not candidates:
        raise RuntimeError('Upscale ciktisi bulunamadi.')
    UPSCALED_PATH = str(candidates[-1])
else:
    UPSCALED_PATH = str(ESR_INPUT)

print('Upscale ciktisi:', UPSCALED_PATH)
display(Image.open(UPSCALED_PATH).resize((512, 512)))

In [ ]:
#@title 6) QR Fusion Icin Kare Hazirla, Arka Plan ve Sharpen Uygula
from PIL import Image, ImageOps, ImageEnhance
from pathlib import Path
import re, shutil, zipfile, os

OUTPUT_DIR = Path('/content/logo_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def hex_to_rgba(value):
    value = value.strip().lstrip('#')
    if not re.fullmatch(r'[0-9a-fA-F]{6}', value):
        raise ValueError('custom_background_color #RRGGBB formatinda olmali.')
    return tuple(int(value[i:i+2], 16) for i in (0, 2, 4)) + (255,)

def fit_square(im, size, mode, bg):
    im = im.convert('RGBA')
    if mode == 'cover_crop':
        return ImageOps.fit(im, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    canvas = Image.new('RGBA', (size, size), bg)
    fitted = ImageOps.contain(im, (size, size), method=Image.Resampling.LANCZOS)
    x = (size - fitted.width) // 2
    y = (size - fitted.height) // 2
    canvas.alpha_composite(fitted, (x, y))
    return canvas

if background_mode == 'white':
    bg = (255, 255, 255, 255)
elif background_mode == 'custom_color':
    bg = hex_to_rgba(custom_background_color)
else:
    bg = (0, 0, 0, 0)

upscaled = Image.open(UPSCALED_PATH).convert('RGBA')
prepared = fit_square(upscaled, int(target_size), fit_mode, bg)

if background_mode != 'keep_transparent':
    solid = Image.new('RGBA', prepared.size, bg)
    solid.alpha_composite(prepared)
    prepared = solid.convert('RGB').convert('RGBA')

if sharpen_after and sharpen_after > 1:
    rgb = prepared.convert('RGB')
    rgb = ImageEnhance.Sharpness(rgb).enhance(float(sharpen_after))
    if background_mode == 'keep_transparent':
        rgb.putalpha(prepared.getchannel('A'))
        prepared = rgb
    else:
        prepared = rgb.convert('RGBA')

logo_upscaled = OUTPUT_DIR / 'logo_upscaled.png'
logo_prepared = OUTPUT_DIR / 'logo_prepared_for_qr.png'
final_placeholder = OUTPUT_DIR / 'final_readable_art_qr_PLACEHOLDER.png'
delivery_zip = OUTPUT_DIR / 'delivery.zip'

shutil.copyfile(UPSCALED_PATH, logo_upscaled)
prepared.save(logo_prepared, 'PNG', optimize=True)
prepared.save(final_placeholder, 'PNG', optimize=True)

settings_text = OUTPUT_DIR / 'settings_used.txt'
settings_text.write_text(f'''Real-ESRGAN Logo QR Preparation Settings\n\nsource={LOGO_PATH}\nupscale_enabled={upscale_enabled}\nupscale_model={upscale_model}\nupscale_factor={upscale_factor}\ntarget_size={target_size}\nfit_mode={fit_mode}\nbackground_mode={background_mode}\ncustom_background_color={custom_background_color}\ndenoise_strength={denoise_strength}\nsharpen_after={sharpen_after}\ntile_size={tile_size}\n''')

with zipfile.ZipFile(delivery_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [logo_upscaled, logo_prepared, final_placeholder, settings_text]:
        zf.write(path, arcname=path.name)

print('Hazir ciktilar:')
print('-', logo_upscaled)
print('-', logo_prepared)
print('-', final_placeholder)
print('-', delivery_zip)
display(prepared.resize((512, 512)))

In [ ]:
#@title 7) ZIP Indir
from google.colab import files
files.download('/content/logo_outputs/delivery.zip')

## Model Secim Notlari

- `realesrgan-x4plus-anime`: flat logo, ikon, maskot, cizim, anime/cartoon tarzi icin varsayilan onerilen model.
- `realesrgan-x4plus`: fotograf, urun gorseli, poster veya gercekci logo icin daha uygun.
- `realesrgan-x2plus`: zaten orta/buyuk logo varsa daha yumusak 2x buyutme icin.
- `realesr-general-x4v3`: JPEG artifact veya noise varsa `denoise_strength` alanini kullanmak icin.

A100 icin `tile_size=768` guvenli varsayilandir. Cok buyuk dosyalarda OOM olursa `512`, maksimum kalite/hiz icin VRAM yeterliyse `0` denenebilir.